# Day 063 | PyTorch — Autograd & Backpropagation

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning | **Author:** Ripon Chandra Malo | **Date:** YYYY-MM-DD

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `requires_grad` | Track operations for gradient computation | Mark soil parameters as trainable |
| Autograd | Automatic differentiation engine | PyTorch computes all derivatives for us |
| `.backward()` | Trigger gradient computation | Compute how loss changes w.r.t. parameters |
| `.grad` | Access computed gradients | Direction to adjust settlement parameters |
| Computational graph | DAG of operations | Forward pass records the recipe for backprop |
| Chain rule | Derivative of composed functions | How nested geotechnical formulas propagate gradients |

---

## Our Geotechnical Problem

We will optimize **foundation settlement parameters** using gradients. Given measured settlement data, we want to find the best elastic modulus (E) and influence factor that minimize prediction error.

```
Elastic settlement:  s = q * B * (1 - v^2) / E * I_f

Where:
  q   = applied pressure (known)
  B   = foundation width (known)
  v   = Poisson's ratio (known)
  E   = elastic modulus (OPTIMIZE THIS)
  I_f = influence factor (OPTIMIZE THIS)
```

We treat E and I_f as learnable parameters and use autograd to find their optimal values.

---

## Part 1: Understanding `requires_grad`

When `requires_grad=True`, PyTorch records every operation on that tensor. This builds a computational graph used for backpropagation.

In [ ]:
# ─────────────────────────────────────────────────────────
# REQUIRES_GRAD — Tracking gradients
# ─────────────────────────────────────────────────────────

import torch
import numpy as np
import matplotlib.pyplot as plt

# Regular tensor — no gradient tracking
depth = torch.tensor([2.0, 5.0, 10.0])
print(f"Regular tensor:      requires_grad = {depth.requires_grad}")

# Tensor with gradient tracking — these are our learnable parameters
E = torch.tensor(25000.0, requires_grad=True)   # Elastic modulus (kPa)
I_f = torch.tensor(0.8, requires_grad=True)     # Influence factor

print(f"E (elastic modulus): requires_grad = {E.requires_grad}")
print(f"I_f (influence):     requires_grad = {I_f.requires_grad}")

# Perform a calculation — PyTorch records the operation
q = torch.tensor(150.0)   # applied pressure (kPa)
B = torch.tensor(2.0)     # foundation width (m)
nu = torch.tensor(0.3)    # Poisson's ratio

# Elastic settlement formula
settlement = q * B * (1 - nu**2) / E * I_f

print(f"\nSettlement = {settlement.item():.6f} m")
print(f"  settlement.requires_grad = {settlement.requires_grad}")
print(f"  settlement.grad_fn = {settlement.grad_fn}")
print("\nThe grad_fn shows this tensor was created by a MulBackward operation.")
print("PyTorch remembers HOW this value was computed — that is the computational graph.")

---

## Part 2: Computing Gradients with `.backward()`

Calling `.backward()` on a scalar computes gradients of that scalar with respect to all tensors that have `requires_grad=True`.

In [ ]:
# ─────────────────────────────────────────────────────────
# BACKWARD PASS — Computing gradients automatically
# ─────────────────────────────────────────────────────────

# Fresh start
E = torch.tensor(25000.0, requires_grad=True)
I_f = torch.tensor(0.8, requires_grad=True)

q = torch.tensor(150.0)
B = torch.tensor(2.0)
nu = torch.tensor(0.3)

# Forward pass
settlement = q * B * (1 - nu**2) / E * I_f

# Define a loss: squared error from measured settlement
measured_settlement = torch.tensor(0.012)  # 12 mm measured
loss = (settlement - measured_settlement) ** 2

print(f"Predicted settlement: {settlement.item():.6f} m")
print(f"Measured settlement:  {measured_settlement.item():.6f} m")
print(f"Loss (MSE):           {loss.item():.10f}")

# Backward pass — compute all gradients
loss.backward()

# Access gradients
print(f"\nGradients:")
print(f"  dLoss/dE   = {E.grad:.10f}")
print(f"  dLoss/dI_f = {I_f.grad:.10f}")
print(f"\nInterpretation:")
print(f"  E.grad < 0 means: increasing E would DECREASE the loss")
print(f"  I_f.grad sign tells us which direction to adjust I_f")

---

## Part 3: The Computational Graph

Every operation creates a node in the graph. The graph is traversed backwards during `.backward()` to compute gradients via the chain rule.

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUALIZE THE COMPUTATIONAL GRAPH STRUCTURE
# ─────────────────────────────────────────────────────────

# Simple example: bearing capacity
c = torch.tensor(25.0, requires_grad=True)    # cohesion (kPa)
phi = torch.tensor(30.0, requires_grad=True)   # friction angle (degrees)

# Terzaghi bearing capacity factors (simplified)
phi_rad = phi * 3.14159 / 180.0
Nc = (torch.tan(phi_rad / 2 + 3.14159 / 4)) ** 2 * torch.exp(3.14159 * torch.tan(phi_rad))

# Ultimate bearing capacity (simplified)
q_ult = c * Nc

print("Computational Graph:")
print(f"  c (leaf)     -> requires_grad={c.requires_grad}, grad_fn={c.grad_fn}")
print(f"  phi (leaf)   -> requires_grad={phi.requires_grad}, grad_fn={phi.grad_fn}")
print(f"  phi_rad      -> grad_fn={phi_rad.grad_fn}")
print(f"  Nc           -> grad_fn={Nc.grad_fn}")
print(f"  q_ult        -> grad_fn={q_ult.grad_fn}")

print(f"\n  q_ult = {q_ult.item():.2f} kPa")

# Compute gradients
q_ult.backward()
print(f"\nGradients:")
print(f"  dq_ult/dc   = {c.grad:.4f}  (how q_ult changes with cohesion)")
print(f"  dq_ult/dphi = {phi.grad:.4f}  (how q_ult changes with friction angle)")

print("\n--- Graph Properties ---")
print("Leaf tensors: c, phi (created by user, not by an operation)")
print("Non-leaf tensors: phi_rad, Nc, q_ult (created by operations)")
print("The graph is destroyed after .backward() (dynamic graph in PyTorch)")

---

## Part 4: Chain Rule Demonstration

The chain rule is the mathematical foundation of backpropagation:

```
dz/dx = dz/dy * dy/dx
```

Let us verify that PyTorch autograd correctly applies the chain rule.

In [ ]:
# ─────────────────────────────────────────────────────────
# CHAIN RULE VERIFICATION
# ─────────────────────────────────────────────────────────

# Example: settlement depends on stress, which depends on depth
# s = f(sigma) = 0.001 * sigma^2
# sigma = g(z) = 18 * z
# ds/dz = ds/dsigma * dsigma/dz = 0.002*sigma * 18 = 0.002*18z * 18

z = torch.tensor(5.0, requires_grad=True)  # depth = 5m

# Forward: z -> sigma -> settlement
sigma = 18.0 * z                     # total stress
settlement = 0.001 * sigma ** 2       # settlement (simplified model)

# Backward
settlement.backward()

# PyTorch's gradient
autograd_result = z.grad.item()

# Manual chain rule
# ds/dsigma = 0.002 * sigma = 0.002 * 90 = 0.18
# dsigma/dz = 18
# ds/dz = 0.18 * 18 = 3.24
sigma_val = 18.0 * 5.0  # = 90
ds_dsigma = 0.002 * sigma_val  # = 0.18
dsigma_dz = 18.0
manual_result = ds_dsigma * dsigma_dz

print(f"Chain Rule Verification:")
print(f"  z = 5.0 m")
print(f"  sigma = 18 * z = {sigma.item():.1f} kPa")
print(f"  s = 0.001 * sigma^2 = {settlement.item():.4f} m")
print(f"")
print(f"  ds/dsigma = 0.002 * sigma = {ds_dsigma:.4f}")
print(f"  dsigma/dz = 18.0")
print(f"  ds/dz = ds/dsigma * dsigma/dz = {ds_dsigma:.4f} * {dsigma_dz:.1f} = {manual_result:.4f}")
print(f"")
print(f"  Autograd result: {autograd_result:.4f}")
print(f"  Manual result:   {manual_result:.4f}")
print(f"  Match: {np.isclose(autograd_result, manual_result)}")

---

## Part 5: Gradient Accumulation and Zeroing

A critical concept: gradients **accumulate** in PyTorch. You must zero them before each backward pass.

In [ ]:
# ─────────────────────────────────────────────────────────
# GRADIENT ACCUMULATION — Why we must zero gradients
# ─────────────────────────────────────────────────────────

x = torch.tensor(3.0, requires_grad=True)

# First backward
y1 = x ** 2
y1.backward()
print(f"After 1st backward: x.grad = {x.grad}  (dy1/dx = 2x = 6)")

# Second backward WITHOUT zeroing
y2 = x ** 2
y2.backward()
print(f"After 2nd backward: x.grad = {x.grad}  (accumulated: 6 + 6 = 12!)")

# This is a BUG if we did not want accumulation!
# Solution: zero the gradient
x.grad.zero_()  # underscore = in-place operation
print(f"After zero_():      x.grad = {x.grad}")

y3 = x ** 2
y3.backward()
print(f"After 3rd backward: x.grad = {x.grad}  (correct: 6)")

print("\n--- Rule ---")
print("Always call optimizer.zero_grad() or param.grad.zero_() before .backward()")
print("Otherwise gradients from previous iterations accumulate and corrupt training!")

---

## Part 6: `torch.no_grad()` — Disabling Gradient Tracking

During evaluation or inference, we do not need gradients. Disabling them saves memory and speeds up computation.

In [ ]:
# ─────────────────────────────────────────────────────────
# NO_GRAD — Disable tracking for inference
# ─────────────────────────────────────────────────────────

E = torch.tensor(30000.0, requires_grad=True)

# With gradient tracking (training mode)
stress = 150.0 * 2.0 / E
print(f"With grad tracking: grad_fn = {stress.grad_fn}")

# Without gradient tracking (evaluation mode)
with torch.no_grad():
    stress_eval = 150.0 * 2.0 / E
    print(f"Without tracking:   grad_fn = {stress_eval.grad_fn}")

# Detach — create a new tensor that shares data but has no grad history
stress_detached = stress.detach()
print(f"Detached:           grad_fn = {stress_detached.grad_fn}")

print("\n--- When to use ---")
print("torch.no_grad():  During evaluation / inference (saves memory)")
print(".detach():        When you need the value without the computation graph")
print("                  (e.g., logging losses, converting to numpy for plots)")

---

## Part 7: Settlement Parameter Optimization with Autograd

Now we use autograd for a real optimization: finding the best elastic modulus and influence factor to match measured settlement data from multiple foundations.

In [ ]:
# ─────────────────────────────────────────────────────────
# GENERATE SYNTHETIC SETTLEMENT DATA
# ─────────────────────────────────────────────────────────

torch.manual_seed(42)

# True parameters (unknown to the optimizer)
E_true = 20000.0      # True elastic modulus (kPa)
I_f_true = 0.88       # True influence factor

# Foundation data: [pressure (kPa), width (m)]
n_foundations = 20
pressures = torch.rand(n_foundations) * 200 + 50     # 50-250 kPa
widths = torch.rand(n_foundations) * 3 + 1           # 1-4 m
nu = 0.3

# Generate "measured" settlements (with noise)
settlements_true = pressures * widths * (1 - nu**2) / E_true * I_f_true
noise = torch.randn(n_foundations) * 0.001  # 1mm noise
settlements_measured = settlements_true + noise

print(f"Generated {n_foundations} foundation settlement measurements")
print(f"True E = {E_true:.0f} kPa, True I_f = {I_f_true:.2f}")
print(f"Pressure range: {pressures.min():.0f} - {pressures.max():.0f} kPa")
print(f"Width range:    {widths.min():.1f} - {widths.max():.1f} m")
print(f"Settlement range: {settlements_measured.min()*1000:.1f} - {settlements_measured.max()*1000:.1f} mm")

In [ ]:
# ─────────────────────────────────────────────────────────
# OPTIMIZE PARAMETERS USING AUTOGRAD
# ─────────────────────────────────────────────────────────

# Initial guesses (deliberately wrong)
E_opt = torch.tensor(35000.0, requires_grad=True)    # guess: 35000 kPa
I_f_opt = torch.tensor(0.5, requires_grad=True)      # guess: 0.5

learning_rate = 500.0  # large LR because E is on the order of 10000s
lr_If = 0.001          # smaller LR for I_f

n_iterations = 500
loss_history = []
E_history = []
If_history = []

for i in range(n_iterations):
    # Zero gradients
    if E_opt.grad is not None:
        E_opt.grad.zero_()
    if I_f_opt.grad is not None:
        I_f_opt.grad.zero_()
    
    # Forward: predict settlements
    settlements_pred = pressures * widths * (1 - nu**2) / E_opt * I_f_opt
    
    # Loss: mean squared error
    loss = torch.mean((settlements_pred - settlements_measured) ** 2)
    
    # Backward: compute gradients
    loss.backward()
    
    # Update parameters (gradient descent)
    with torch.no_grad():
        E_opt -= learning_rate * E_opt.grad
        I_f_opt -= lr_If * I_f_opt.grad
    
    # Record history
    loss_history.append(loss.item())
    E_history.append(E_opt.item())
    If_history.append(I_f_opt.item())
    
    if i % 50 == 0 or i == n_iterations - 1:
        print(f"Iter {i:3d} | Loss: {loss.item():.2e} | "
              f"E: {E_opt.item():.0f} kPa | I_f: {I_f_opt.item():.4f}")

print(f"\n--- Results ---")
print(f"True:      E = {E_true:.0f} kPa,  I_f = {I_f_true:.4f}")
print(f"Optimized: E = {E_opt.item():.0f} kPa,  I_f = {I_f_opt.item():.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUALIZE OPTIMIZATION PROGRESS
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Loss curve
axes[0].plot(loss_history, color='steelblue', linewidth=1.5)
axes[0].set_yscale('log')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('MSE Loss (log scale)', fontsize=12)
axes[0].set_title('Loss Curve', fontsize=13)
axes[0].grid(True, alpha=0.3)

# E convergence
axes[1].plot(E_history, color='coral', linewidth=1.5, label='Optimized E')
axes[1].axhline(y=E_true, color='green', linestyle='--', linewidth=2, label=f'True E = {E_true:.0f}')
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Elastic Modulus E (kPa)', fontsize=12)
axes[1].set_title('E Convergence', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# I_f convergence
axes[2].plot(If_history, color='orchid', linewidth=1.5, label='Optimized I_f')
axes[2].axhline(y=I_f_true, color='green', linestyle='--', linewidth=2, label=f'True I_f = {I_f_true:.2f}')
axes[2].set_xlabel('Iteration', fontsize=12)
axes[2].set_ylabel('Influence Factor I_f', fontsize=12)
axes[2].set_title('I_f Convergence', fontsize=13)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Settlement Parameter Optimization via Autograd', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# COMPARE PREDICTED VS MEASURED SETTLEMENTS
# ─────────────────────────────────────────────────────────

with torch.no_grad():
    final_pred = pressures * widths * (1 - nu**2) / E_opt * I_f_opt

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(settlements_measured.numpy() * 1000, final_pred.numpy() * 1000,
           alpha=0.7, edgecolors='k', linewidth=0.5, s=60)
lim = [0, max(settlements_measured.max().item(), final_pred.max().item()) * 1000 * 1.1]
ax.plot(lim, lim, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Measured Settlement (mm)', fontsize=12)
ax.set_ylabel('Predicted Settlement (mm)', fontsize=12)
ax.set_title('Optimized Settlement Predictions', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# Compute R-squared
ss_res = torch.sum((settlements_measured - final_pred) ** 2)
ss_tot = torch.sum((settlements_measured - settlements_measured.mean()) ** 2)
r2 = (1 - ss_res / ss_tot).item()
print(f"R-squared = {r2:.6f}")

---

## Part 8: Gradients of Vector-Valued Functions

When the output is not a scalar, we need to provide a `gradient` argument (Jacobian-vector product). In practice, we always reduce to a scalar loss.

In [ ]:
# ─────────────────────────────────────────────────────────
# VECTOR-VALUED GRADIENTS
# ─────────────────────────────────────────────────────────

# Soil stiffness parameter
k = torch.tensor(2.0, requires_grad=True)

# Multiple loads -> multiple settlements
loads = torch.tensor([50.0, 100.0, 200.0, 400.0])  # kN
settlements = loads / (k * 1000)  # simplified: s = P / (k * 1000)

print(f"Settlements: {settlements.tolist()}")

# Option 1: Reduce to scalar first (most common)
total_settlement = settlements.sum()
total_settlement.backward()
print(f"\nGradient of sum(settlements) w.r.t. k: {k.grad:.6f}")

# Option 2: Provide gradient vector (Jacobian-vector product)
k2 = torch.tensor(2.0, requires_grad=True)
s2 = loads / (k2 * 1000)
# Weight each settlement equally
s2.backward(torch.ones_like(s2))  # same as summing then backward
print(f"With gradient vector:                  {k2.grad:.6f}")

print("\nIn practice, we always compute a scalar loss (MSE, MAE, etc.)")
print("and call .backward() on that scalar — autograd handles the rest.")

---

## Practice Exercises

### Exercise 1: Optimize Consolidation Parameters
Given consolidation data (time, settlement), optimize Cv (coefficient of consolidation) to match the curve `U = 1 - exp(-Cv * t / H^2)`. Use autograd.

### Exercise 2: Multi-Parameter Bearing Capacity
Optimize both cohesion (c) and friction angle (phi) to match bearing capacity test data using `q_ult = c * Nc + 0.5 * gamma * B * Ngamma`.

### Exercise 3: Verify Chain Rule for a 3-Link Chain
Compute `f(x) = sin(exp(x^2))` at x=1. Verify autograd matches your manual chain rule calculation.

In [ ]:
# EXERCISE 1: Consolidation parameter optimization
# Cv = torch.tensor(0.5, requires_grad=True)
# H = 5.0  # drainage path
# t = torch.linspace(0, 100, 50)
# U_measured = 1 - torch.exp(-0.8 * t / H**2) + noise
# Optimize Cv to minimize MSE between predicted and measured U

# EXERCISE 2: Multi-parameter bearing capacity
# c_opt = torch.tensor(20.0, requires_grad=True)
# phi_opt = torch.tensor(25.0, requires_grad=True)
# Use autograd to optimize both simultaneously

# EXERCISE 3: Chain rule verification
# x = torch.tensor(1.0, requires_grad=True)
# f = torch.sin(torch.exp(x**2))
# f.backward()
# Compare x.grad with manual: cos(exp(1)) * exp(1) * 2

---

## Key Takeaways

1. **`requires_grad=True` enables automatic differentiation.** PyTorch records all operations to build a computational graph.

2. **`.backward()` computes gradients** via the chain rule, traversing the graph from output to inputs.

3. **Gradients accumulate by default.** Always zero them before each backward pass with `.grad.zero_()` or `optimizer.zero_grad()`.

4. **`torch.no_grad()` saves memory during inference.** Use it for evaluation, logging, or converting to numpy.

5. **Autograd replaces manual backpropagation.** What took 30 lines in Day 061 is now just `loss.backward()` — PyTorch handles the chain rule automatically.

6. **Any differentiable geotechnical formula can be optimized with autograd.** Settlement, bearing capacity, consolidation — if you can write it in PyTorch, you can optimize it.

---

## How This Connects to ML

| Today (Autograd) | Neural Network Training |
|------------------|------------------------|
| `requires_grad=True` | All `nn.Module` parameters have this by default |
| `loss.backward()` | Same call in every training loop |
| Manual `W -= lr * W.grad` | `optimizer.step()` does this for all parameters |
| `param.grad.zero_()` | `optimizer.zero_grad()` |
| `torch.no_grad()` | Used in `model.eval()` mode for validation |
| Chain rule through formulas | Chain rule through network layers |

**Tomorrow:** We combine tensors + autograd to build our first neural network using `nn.Module` and `nn.Linear`.

---

*Day 063 of 100 — We let PyTorch do the calculus while we focus on the geotechnical engineering!*